# GPU check for paged-kv-cache

Runs the CUDA backend's tests on a real GPU. Each test compares the GPU kernels against `CpuBackend`, the reference implementation, element by element.

1. **Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**.
2. The last cell prints the test results. The first run takes a few minutes, most of it compiling.

On Kaggle, turn on a GPU accelerator and internet access in the notebook settings first.

In [6]:
# Set BRANCH to test a pull request before it is merged.
REPO = "https://github.com/SohamDutta02/paged-kv-cache"
BRANCH = "paged-attention"

In [7]:
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv

name, driver_version, memory.total [MiB]
Tesla T4, 580.82.07, 15360 MiB


In [8]:
import glob, os, subprocess

!curl -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal -q
os.environ["PATH"] = os.path.expanduser("~/.cargo/bin") + ":/usr/local/cuda/bin:" + os.environ["PATH"]

# cudarc loads NVRTC at runtime. Use the toolkit's copy if there is one, else install it.
nvrtc = sorted(glob.glob("/usr/local/cuda*/lib64/libnvrtc.so*"))
if nvrtc:
    lib = os.path.dirname(nvrtc[0])
else:
    subprocess.run(["pip", "install", "-q", "nvidia-cuda-nvrtc-cu12"], check=True)
    import nvidia.cuda_nvrtc
    lib = os.path.join(nvidia.cuda_nvrtc.__path__[0], "lib")
    if not os.path.exists(os.path.join(lib, "libnvrtc.so")):
        os.symlink("libnvrtc.so.12", os.path.join(lib, "libnvrtc.so"))
os.environ["LD_LIBRARY_PATH"] = lib + ":" + os.environ.get("LD_LIBRARY_PATH", "")
print("NVRTC from", lib)
!cargo --version

warn: it looks like you have an existing rustup settings file at:
warn: /root/.rustup/settings.toml
warn: Updating existing toolchain, profile choice will be ignored

  stable-x86_64-unknown-linux-gnu unchanged - rustc 1.99.0 (b940084d7 2026-09-28)


Rust is installed now. Great!

To get started you may need to restart your current shell.
This would reload your PATH environment variable to include
Cargo's bin directory ($HOME/.cargo/bin).

To configure your current shell, you need to source the
corresponding env file under $HOME/.cargo.

Consider running the right command for your shell (note the leading DOT):
. "$HOME/.cargo/env" # For sh/ash/dash/pdksh/bash
cargo:rerun-if-env-changed=CC_x86_64-unknown-linux-gnu
CC_x86_64-unknown-linux-gnu = None
cargo:rerun-if-env-changed=CC_x86_64_unknown_linux_gnu
CC_x86_64_unknown_linux_gnu = None
cargo:rerun-if-env-changed=HOST_CC
HOST_CC = None
cargo:rerun-if-env-changed=CC
CC = None
cargo:rerun-if-env-changed=CC_ENABLE_DEBUG_OUTPUT
cargo:rerun-

In [9]:
!rm -rf paged-kv-cache && git clone -q --depth 1 --branch {BRANCH} {REPO}
!cd paged-kv-cache && git log -1 --format='Testing %h: %s'

Testing 30dd21a: Document paged attention


In [ ]:
!cd paged-kv-cache && cargo test --release -p paged-kv-cuda --features cuda -- --test-threads=1 2>&1 | grep -vE '^\s+(Compiling|Downloaded|Downloading)'